## Embed and Plot the UMAP Pilot Controls

Lead     : `Pixel / Pixeldom4`

Issue    : [Github Issue #75](https://github.com/petadex/igem-toronto/issues/75) — _Embed and Plot the UMAP Pilot Controls_

Start    : `2026-05-29`

Complete : `2026-05-29`

Files    : `~/resources/260529_issue75_esm_controls_umap/` — embedding-bundle + UMAP scripts and plot

S3 files : `s3://petadex/esm_embeddings/esm_controls/` — `controls.npz`, `family_embeddings.npz`, `umap_output.csv`

Depends on: [#34 (control generation)](https://github.com/petadex/igem-toronto/issues/34) · [#32 (ESM-2 pipeline)](https://github.com/petadex/igem-toronto/issues/32)

## Introduction

Issue #34 generated six control sequence sets from the 64,730 PETase family representatives — random strings (length matched / empirical / 95th-percentile), shuffled centroids, real-sequence fragments, and random UniProt proteins — totalling 517,840 sequences. On their own these are just FASTA-like CSVs; they only become controls once they live in the same embedding space as the real centroids.

This issue (#75) is the downstream half: embed every control through the **identical** ESM-2 pipeline used for the real centroids in #32, then fit a single UMAP over the union of controls and real centroids so all 582,570 points share one coordinate system. The shared fit is what lets us ask whether the apparent clusters in the #13 landscape are meaningful or are artefacts of amino-acid composition.

Two questions decide the outcome. **(1)** Do real proteins occupy a region distinct from random amino-acid strings? If random sequences land on the real centroids, the landscape is noise. **(2)** Does the embedding encode amino-acid *composition* or sequence *order / structure*? The shuffled control — identical length and composition to a real sequence, only the order destroyed — is the discriminating test: a composition-only encoder would map a sequence and its shuffle to the same point.

## Objectives

1. Embed the six #34 control sets through the #32 ESM-2 pipeline so controls and real centroids are directly comparable.
2. Bundle the embedding shard into a single labelled `.npz`, recovering the per-sequence control variant.
3. Co-UMAP the 517,840 controls with the 64,730 real centroids in one coordinate system and plot.
4. Interpret the projection: do random sequences separate from real, and does the shuffled control sit apart (order vs. composition)?

---

## Materials and Methods

### System

- **Encoder**: `facebook/esm2_t30_150M_UR50D` (150M params, 640-dim), mean-pooled over real residues, float16 — identical to the #32 run on the real centroids.
- **Reducer**: `umap-learn`, `n_neighbors=15`, `min_dist=0.1`, `metric=euclidean`, `random_state=42`.

### Inputs

```bash
# From issue #34: six control CSVs (family_id, sequence), 517,840 rows total
# From issue #32: family_embeddings.npz — 64,730 real centroids (640-dim, float32)
aws s3 cp s3://petadex/esm_embeddings/esm_landscape/family_embeddings.npz ./
```

### Pipeline

**Step 1 — Embed (issue #32 pipeline).** The control CSVs are embedded with the same ESM-2 model and mean-pooling as the real centroids, producing a single float16 shard and a parallel ids file:

```
controls.npy        (517840, 640) float16
controls_ids.txt    517840 ids, e.g.  rand_95th_family_0__0
```

**Step 2 — Bundle into a labelled `.npz`** (`npy_to_npz.py`). The shard carries no labels; the control variant is recovered from each id by stripping the trailing global index and local id:

```bash
python3 scripts/npy_to_npz.py --emb controls.npy --ids controls_ids.txt --out controls.npz
# rand_95th_family_0__0  -> variant 'rand_95th_family'
# rand_fragment_21633__194188 -> 'rand_fragment'
# shuffled_64918__517839 -> 'shuffled'
```

The script prints per-variant counts — verify them (64,730 each; 194,190 fragments; 517,840 total) before continuing.

**Step 3 — Co-UMAP with the real centroids** (`run_umap.py`). Both `.npz` files are stacked and fit together. `family_embeddings.npz` has no `variants` column so it takes the `--labels` value `real`; `controls.npz` is labelled by its own `variants` array:

```bash
python3 scripts/run_umap.py family_embeddings.npz controls.npz \
    --labels real --output umap_output.csv --plot plots/umap.png
# Fitting UMAP on 582,570 sequences (640D) ...
```

### Procedural pitfalls

Annotated because they cost time or could silently corrupt the result:

1. **Row order is the only key.** `npy_to_npz.py` pairs embedding row *i* with id line *i* by position — there is no join key. If the embedding job finishes batches out of order or drops a sequence (e.g. one over a length cap), the ids and embeddings desync and **every point is mislabelled**. The script hard-fails on a count *mismatch*, but an equal-count reorder passes silently. Keep embedding output strictly in input order.

2. **Variant labels are inferred by regex, not stored.** `variant_of()` strips `__<globalidx>` then `_<localid>`, assuming ids look like `<variant>_<localid>__<globalidx>`. An unexpected underscore in an accession or `family_id` mis-splits the variant. Always sanity-check the printed per-variant counts.

3. **Mixed dtypes across files.** `controls.npz` is float16, `family_embeddings.npz` is float32. `run_umap.py` casts both to float32 before `vstack`, so it is handled — but a hand-rolled stack on mixed dtypes will upcast/copy and can spike memory.

4. **`np.savez` materialises the memmap.** `npy_to_npz.py` opens the `.npy` as a memmap but `savez` pulls the full 632 MB into RAM to write. `--compress` is near-useless on high-entropy float16 (barely shrinks, ~2× slower) — keep the default uncompressed.

5. **Single joint UMAP fit, controls outnumber real ~8:1.** All 582,570 points are fit together for one shared coordinate system, so the 517,840 controls dominate the manifold and shape where the real centroids land. Intended — but do not read absolute inter-cluster distances as if the real points were embedded alone.

6. **`random_state=42` disables UMAP parallelism.** Reproducible but single-threaded; fitting 582k × 640 is slow and memory-heavy. Budget the time — it is not hung.

7. **Plot overdraw hides minority sets.** Groups are scattered in alphabetical order at `alpha=0.4`; whatever draws last sits on top. If a control looks absent, check `umap_output.csv` before concluding overlap.

8. **Random-UniProt under-delivery is silent.** The #34 fetch skips failed REST batches and entries lacking a sequence, so the count can fall short of *n*. Verify the `rand_uniprot` count at generation, not after embedding.

## Results & Discussion

![UMAP of real family centroids and all six controls](../resources/260529_issue75_esm_controls_umap/plots/umap.png)

**The real centroids occupy distinct, dense islands (brown).** The PETase family representatives form several compact clusters rather than spreading uniformly — the landscape structure survives the control comparison.

**Random sequences collapse into their own territory, away from real proteins.** The `rand_matched` set (red) condenses into a single tight blob in the lower-right, well separated from every real island. The length-varied random sets (`rand_empirical`, `rand_95th`) behave the same way. ESM-2 does **not** confuse uniform-random amino-acid strings with real proteins, regardless of how their lengths are chosen — exactly the negative-control behaviour we wanted to confirm.

**Shuffled sequences form a distinct cluster from the real centroids (pink, upper-centre).** This is the headline result. The shuffled control has *identical length and amino-acid composition* to the real sequences — only the order is destroyed — yet it lands in its own region rather than on top of the real islands. **The embeddings are therefore encoding sequence order / structure, not merely amino-acid composition.** A bag-of-amino-acids encoder would have placed each shuffle on its real counterpart; ESM-2 does not. This justifies treating the landscape as structurally meaningful.

**Fragments stay near the real islands (green).** The 30/60/90% fragments cluster adjacent to and overlapping the real centroids — partial real sequence retains a real-like embedding, with fragments forming satellite groups around the parent clusters.

**Random UniProt proteins spread broadly across the plot (purple).** Generic real proteins occupy a wide, diffuse region rather than a single blob, reflecting their diversity — and the PETase centroids sit as distinct, concentrated islands *within* that broader real-protein space, consistent with PETases being a specific structural family.

### Summary

| Control | Behaviour vs. real | Interpretation |
|---|---|---|
| Random matched / empirical / 95th | Tight blob, fully separated | Random ≠ real; landscape is not noise |
| Shuffled | Distinct cluster, not co-located | **Embeddings encode order/structure, not just composition** |
| Fragments | Adjacent to / overlapping real islands | Partial real sequence reads as real |
| Random UniProt | Broad, diffuse spread | PETases are a distinct family within real-protein space |

### Follow-up questions

- **UMAP variance**: run multiple seeds and measure cluster stability — quantify how much of the layout is projection variance vs. signal.
- **Random UniProt domains**: pull domain-level sequences (not just full proteins) to test whether domains cluster together, probing the structural basis of the clustering.
- **Characterized enzymes**: overlay known active vs. inactive (dead) PETases as labelled points to check whether activity separates in the landscape.
- **Taxonomic controls**: sample proteins from labelled taxa to test whether any apparent clustering is taxonomy-driven (cf. the issue #74 phylum probe).